In [1]:
import math

## Reading the Sequences
The idea is to read aligned sequences from a text file, calculate a position-specific scoring matrix from them, and then score example sequences against that matrix. The first part reads one sequence per line. 

In [2]:
def read_sequences(filename):
    sequences = []
    with open(filename) as file:
        for line in file:
            line = line.strip().upper()
            if line:
                sequences.append(line)
    if not sequences:
        raise ValueError("Input file does not contain any sequences")
    expected_length = len(sequences[0])
    for seq in sequences:
        if len(seq) != expected_length:
            raise ValueError("All sequences must have the same length")
    return sequences


def get_alphabet(sequences):
    return sorted(set("".join(sequences)))

In [3]:
read_sequences("../seq1.txt")

['ACGTA',
 'ACGTT',
 'ACGAA',
 'TCGTA',
 'ACCTA',
 'AGGTA',
 'ACGCA',
 'ACGGA',
 'CCGTA',
 'GCGTA',
 'ATGTA',
 'AAGTA',
 'ACATA',
 'ACTTA',
 'ACGTC',
 'ACGTG',
 'ACGAT',
 'TCGTT',
 'ACCAA',
 'GGGTA',
 'ATCTA',
 'ACATT',
 'CCGTT',
 'GCGAA']

## Assertion Helper

This helper is used for direct list inputs. It raises `TypeError` when the input is not a list of strings, and `ValueError` when the strings do not all have the same length.

In [4]:
def assert_equal_length_strings(strings):
    if not isinstance(strings, list):
        raise TypeError("Expected a list of strings")

    for value in strings:
        if not isinstance(value, str):
            raise TypeError("Expected a list of strings")

    if strings:
        expected_length = len(strings[0])
        for value in strings:
            if len(value) != expected_length:
                raise ValueError("All strings must have the same length")

## Count Matrix

The count matrix then stores how often every letter occurs at every position:

$$
C_{a,j} = \sum_{i=1}^{m}
\begin{cases}
1, & \text{if sequence } i \text{ has letter } a \text{ at position } j \\
0, & \text{otherwise}
\end{cases}
$$

Each column in the matrix is a dictionary with one entry per alphabet letter.

In [5]:
def get_count_matrix(sequences, alphabet):
    length = len(sequences[0])
    count_matrix = []
    assert_equal_length_strings(sequences)
    for position in range(length):
        counts = {}
        for letter in alphabet:
            counts[letter] = 0
        for seq in sequences:
            letter = seq[position]
            counts[letter] += 1
        count_matrix.append(counts)

    return count_matrix

In [6]:
get_count_matrix(
    ["XXXXY", "XYXYX"],
    ["X", "Y", "Z"]
)

[{'X': 2, 'Y': 0, 'Z': 0},
 {'X': 1, 'Y': 1, 'Z': 0},
 {'X': 2, 'Y': 0, 'Z': 0},
 {'X': 1, 'Y': 1, 'Z': 0},
 {'X': 1, 'Y': 1, 'Z': 0}]

In [7]:
# should fail with ValueError
try:
    get_count_matrix(
        ["XXXXYZ", "XYXYX"],
        ["X", "Y", "Z"]
    )
except ValueError:
    print("failed successfully")
else:
    raise AssertionError("Expected ValueError")

failed successfully


## Frequency Matrix

The frequency matrix is calculated from the count matrix:

$$
F_{a,j} = \frac{C_{a,j}}{m}
$$

In [8]:
def get_frequency_matrix(count_matrix):
    frequency_matrix = []
    for counts in count_matrix:
        total = sum(counts.values())
        frequencies = {}
        for letter in counts:
            frequencies[letter] = counts[letter] / total
        frequency_matrix.append(frequencies)
    return frequency_matrix

In [9]:
count_matrix = get_count_matrix(
    ["XXX", "YYY", "XYX", "YYX"],
    ["X", "Y"]
)
get_frequency_matrix(count_matrix)

[{'X': 0.5, 'Y': 0.5}, {'X': 0.25, 'Y': 0.75}, {'X': 0.75, 'Y': 0.25}]

## Scoring Matrix

The final position-specific score is the natural logarithm of the observed frequency divided by the background probability:

$$
M_{a,j} =
\begin{cases}
\ln\left(\frac{F_{a,j}}{B_a}\right), & \text{if } F_{a,j} > 0 \\
-\infty, & \text{if } F_{a,j} = 0
\end{cases}
$$

The corresponding code uses $\verb|math.log|$ for the natural logarithm. If a letter was never observed in a column, the score is set to $\verb|-inf|$.

In [10]:
def get_scoring_matrix(frequency_matrix, background_probability):
    scoring_matrix = []
    for frequencies in frequency_matrix:
        scores = {}

        for letter in frequencies:
            if frequencies[letter] == 0:
                scores[letter] = float("-inf")
            else:
                scores[letter] = math.log(
                    frequencies[letter] / background_probability[letter]
                )
        scoring_matrix.append(scores)
    return scoring_matrix

In [11]:
cnt_matrix = get_count_matrix(
    ["XXYXX", "YYYYY", "XYXXX", "YYXYX", "YXXYX", "XXXXZ"],
    ["X", "Y", "Z"]
)
cnt_matrix

[{'X': 3, 'Y': 3, 'Z': 0},
 {'X': 3, 'Y': 3, 'Z': 0},
 {'X': 4, 'Y': 2, 'Z': 0},
 {'X': 3, 'Y': 3, 'Z': 0},
 {'X': 4, 'Y': 1, 'Z': 1}]

In [12]:
freq_matrix = get_frequency_matrix(cnt_matrix)
get_scoring_matrix(freq_matrix, {"X":1/3, "Y":1/3, "Z":1/3})

[{'X': 0.4054651081081644, 'Y': 0.4054651081081644, 'Z': -inf},
 {'X': 0.4054651081081644, 'Y': 0.4054651081081644, 'Z': -inf},
 {'X': 0.6931471805599453, 'Y': 0.0, 'Z': -inf},
 {'X': 0.4054651081081644, 'Y': 0.4054651081081644, 'Z': -inf},
 {'X': 0.6931471805599453, 'Y': -0.6931471805599453, 'Z': -0.6931471805599453}]

## Sequence Scoring

The score of a new sequence \(x\) is the sum of the score belonging to the observed letter at each position:

$$
S(x) = \sum_{j=1}^{L} M_{x_j,j}
$$

The scoring function implements this by adding exactly one matrix value per position. It also checks that the query sequence has the same length as the scoring matrix.

In [13]:
def score_sequence(sequence, scoring_matrix):
    if len(sequence) != len(scoring_matrix):
        raise ValueError("Sequence length must match the scoring matrix length")

    score = 0
    for position, letter in enumerate(sequence):
        if letter not in scoring_matrix[position]:
            raise ValueError(f"Unknown letter {letter} at position {position + 1}")
        score += scoring_matrix[position][letter]
    return score

In [14]:
score_sequence("XYXYX", get_scoring_matrix(freq_matrix, {"X":1/3, "Y":1/3, "Z":1/3}))

2.6026896854443837

## Testing Helper

I also have an output helper that prints the internal matrix representation as markdown-style tables. 

In [15]:
def display_matrix(matrix, alphabet, decimals=None):
    headers = []
    for column in matrix:
        headers.append(max(alphabet, key=lambda letter: column[letter]))

    header = [""] + headers
    separator = ["---"] + ["---:" for _ in headers]
    print("| " + " | ".join(header) + " |")
    print("| " + " | ".join(separator) + " |")

    for letter in alphabet:
        row = [letter]
        for column in matrix:
            value = column[letter]
            if math.isinf(value):
                row.append("-inf")
            elif decimals is None:
                row.append(str(value))
            else:
                row.append(f"{value:.{decimals}f}")
        print("| " + " | ".join(row) + " |")


def run_test(filename, query_sequences):
    sequences = read_sequences(filename)
    alphabet = get_alphabet(sequences)
    background_probability = {}
    for letter in alphabet:
        background_probability[letter] = 1 / len(alphabet)

    count_matrix = get_count_matrix(sequences, alphabet)
    frequency_matrix = get_frequency_matrix(count_matrix)
    scoring_matrix = get_scoring_matrix(frequency_matrix, background_probability)

    print(f"Count matrix from {filename}")
    display_matrix(count_matrix, alphabet)
    print()
    print(f"Frequency matrix from {filename}")
    display_matrix(frequency_matrix, alphabet, 2)
    print()
    print(f"Scoring matrix from {filename}")
    display_matrix(scoring_matrix, alphabet, 3)
    print()
    print("Example scores")

    for sequence in query_sequences:
        score = score_sequence(sequence, scoring_matrix)
        print(f"{sequence}: {score:.3f}")

## Test Inputs

### seq2.txt

In [16]:
run_test("../seq2.txt", ["ACGTA", "ACGTT"])

Count matrix from ../seq2.txt
|  | A | C | G | T | A |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 5 | 0 | 0 | 1 | 5 |
| C | 0 | 5 | 1 | 0 | 0 |
| G | 0 | 1 | 5 | 0 | 0 |
| T | 1 | 0 | 0 | 5 | 1 |

Frequency matrix from ../seq2.txt
|  | A | C | G | T | A |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 0.83 | 0.00 | 0.00 | 0.17 | 0.83 |
| C | 0.00 | 0.83 | 0.17 | 0.00 | 0.00 |
| G | 0.00 | 0.17 | 0.83 | 0.00 | 0.00 |
| T | 0.17 | 0.00 | 0.00 | 0.83 | 0.17 |

Scoring matrix from ../seq2.txt
|  | A | C | G | T | A |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 1.204 | -inf | -inf | -0.405 | 1.204 |
| C | -inf | 1.204 | -0.405 | -inf | -inf |
| G | -inf | -0.405 | 1.204 | -inf | -inf |
| T | -0.405 | -inf | -inf | 1.204 | -0.405 |

Example scores
ACGTA: 6.020
ACGTT: 4.410


### seq1.txt

In [17]:
run_test("../seq1.txt", ["ACGTA", "ACGTT"])

Count matrix from ../seq1.txt
|  | A | C | G | T | A |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 17 | 1 | 2 | 4 | 17 |
| C | 2 | 19 | 3 | 1 | 1 |
| G | 3 | 2 | 18 | 1 | 1 |
| T | 2 | 2 | 1 | 18 | 5 |

Frequency matrix from ../seq1.txt
|  | A | C | G | T | A |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 0.71 | 0.04 | 0.08 | 0.17 | 0.71 |
| C | 0.08 | 0.79 | 0.12 | 0.04 | 0.04 |
| G | 0.12 | 0.08 | 0.75 | 0.04 | 0.04 |
| T | 0.08 | 0.08 | 0.04 | 0.75 | 0.21 |

Scoring matrix from ../seq1.txt
|  | A | C | G | T | A |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 1.041 | -1.792 | -1.099 | -0.405 | 1.041 |
| C | -1.099 | 1.153 | -0.693 | -1.792 | -1.792 |
| G | -0.693 | -1.099 | 1.099 | -1.792 | -1.792 |
| T | -1.099 | -1.099 | -1.792 | 1.099 | -0.182 |

Example scores
ACGTA: 5.433
ACGTT: 4.209


### seq3.txt

In [18]:
run_test("../seq3.txt", ["MKTWF", "MKTWY", "MKAWF"])

Count matrix from ../seq3.txt
|  | M | K | T | W | F |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 0 | 0 | 1 | 0 | 0 |
| F | 0 | 0 | 0 | 1 | 5 |
| K | 0 | 5 | 0 | 0 | 0 |
| L | 1 | 0 | 0 | 0 | 0 |
| M | 5 | 0 | 0 | 0 | 0 |
| R | 0 | 1 | 0 | 0 | 0 |
| T | 0 | 0 | 5 | 0 | 0 |
| W | 0 | 0 | 0 | 5 | 0 |
| Y | 0 | 0 | 0 | 0 | 1 |

Frequency matrix from ../seq3.txt
|  | M | K | T | W | F |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | 0.00 | 0.00 | 0.17 | 0.00 | 0.00 |
| F | 0.00 | 0.00 | 0.00 | 0.17 | 0.83 |
| K | 0.00 | 0.83 | 0.00 | 0.00 | 0.00 |
| L | 0.17 | 0.00 | 0.00 | 0.00 | 0.00 |
| M | 0.83 | 0.00 | 0.00 | 0.00 | 0.00 |
| R | 0.00 | 0.17 | 0.00 | 0.00 | 0.00 |
| T | 0.00 | 0.00 | 0.83 | 0.00 | 0.00 |
| W | 0.00 | 0.00 | 0.00 | 0.83 | 0.00 |
| Y | 0.00 | 0.00 | 0.00 | 0.00 | 0.17 |

Scoring matrix from ../seq3.txt
|  | M | K | T | W | F |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | -inf | -inf | 0.405 | -inf | -inf |
| F | -inf | -inf | -inf | 0.405 | 2.015 |
| K | -inf

## Manual Workthrough

### seq2.txt

We will do a manual trace for the first example:

| Index | Sequence |
|--:|--|
| 1 | `ACGTA` |
| 2 | `ACGTT` |
| 3 | `ACGAA` |
| 4 | `TCGTA` |
| 5 | `ACCTA` |
| 6 | `AGGTA` |

The count matrix stores one dictionary per position. Written as a table, the same values are:

| Letter | 1 (A) | 2 (C) | 3 (G) | 4 (T) | 5 (A) |
|--|--:|--:|--:|--:|--:|
| A | 5 | 0 | 0 | 1 | 5 |
| C | 0 | 5 | 1 | 0 | 0 |
| G | 0 | 1 | 5 | 0 | 0 |
| T | 1 | 0 | 0 | 5 | 1 |

The frequency matrix divides every count by the number of sequences. Here $m=6$:

$$
F_{A,1}=\frac{5}{6}=0.83
$$

$$
F_{T,1}=\frac{1}{6}=0.17
$$

| Letter | 1 (A) | 2 (C) | 3 (G) | 4 (T) | 5 (A) |
|--|--:|--:|--:|--:|--:|
| A | 0.83 | 0.00 | 0.00 | 0.17 | 0.83 |
| C | 0.00 | 0.83 | 0.17 | 0.00 | 0.00 |
| G | 0.00 | 0.17 | 0.83 | 0.00 | 0.00 |
| T | 0.17 | 0.00 | 0.00 | 0.83 | 0.17 |

The background probability is uniform, so for DNA $B_a=0.25$. A common letter gets a positive log score:

$$
M_{A,1}=\ln\left(\frac{5/6}{0.25}\right)=\ln\left(\frac{10}{3}\right)\approx 1.204
$$

A less common observed letter gets a negative score:

$$
M_{T,1}=\ln\left(\frac{1/6}{0.25}\right)=\ln\left(\frac{2}{3}\right)\approx -0.405
$$

A letter that never occurred at a position gets $-\infty$.

| Letter | 1 (A) | 2 (C) | 3 (G) | 4 (T) | 5 (A) |
|--|--:|--:|--:|--:|--:|
| A | 1.204 | $-\infty$ | $-\infty$ | -0.405 | 1.204 |
| C | $-\infty$ | 1.204 | -0.405 | $-\infty$ | $-\infty$ |
| G | $-\infty$ | -0.405 | 1.204 | $-\infty$ | $-\infty$ |
| T | -0.405 | $-\infty$ | $-\infty$ | 1.204 | -0.405 |

For `ACGTA`, all five letters are the most frequent letters at their positions:

$$
S(\texttt{ACGTA})=1.204+1.204+1.204+1.204+1.204=6.020
$$

For `ACGTT`, the last position uses `T` instead of the more frequent `A`:

$$
S(\texttt{ACGTT})=1.204+1.204+1.204+1.204-0.405=4.410
$$

| Sequence | Score |
|--|--:|
| `ACGTA` | 6.020 |
| `ACGTT` | 4.410 |

which is the same result as the code.